# Phase 3 — Lean LangChain Agent (on top of Decision Engine outputs)

This notebook builds a **simple, context-tuned agent** that operates **only on your Decision Engine outputs** (no re-simulation).
It performs deterministic checks (EUR per AU, budget, retention) and uses a **LangChain agent** to deliver an executive **robustness verdict**.

**Inputs** (export these once from your engine):
- `scenario_compare.csv` with columns: `option, delta_net_eur, delta_grr_pp, delta_nrr_pp, effort_au`
- `greedy_summary.json` with keys: `budget_used_au, delta_net_eur, delta_grr_pp`

**Output**: `agentic_checks.json` with summary, stress-test verdicts, and a robust recommendation.


In [31]:
# 1) Config and inputs
import os, json, math
import pandas as pd

MAX_PAYBACK_MONTHS = 18  # payback gate <= 18 months (engine realism)
BUDGET_CAP_AU = float(os.getenv('BUDGET_CAP_AU', '50.0'))
SCENARIO_CSV = os.getenv('SCENARIO_CSV', 'scenario_compare.csv')
GREEDY_JSON  = os.getenv('GREEDY_JSON',  'greedy_summary.json')

scenarios_df = pd.read_csv(SCENARIO_CSV)
with open(GREEDY_JSON, 'r') as f:
    greedy_summary = json.load(f)

# Normalize schema
name_col = None
for c in scenarios_df.columns:
    if str(c).lower() in ('option','scenario','name'):
        name_col = c
        break
if name_col is None:
    name_col = scenarios_df.columns[0]
scenarios_df.rename(columns={name_col:'option'}, inplace=True)
required = {'option','delta_net_eur','delta_grr_pp','delta_nrr_pp','effort_au'}
missing = required - set(map(str, scenarios_df.columns))
if missing:
    raise ValueError(f'Missing columns in {SCENARIO_CSV}: {missing}')

scenarios_df


,option,delta_net_eur,delta_grr_pp,delta_nrr_pp,effort_au
0,Retention-first,20882.340,0.426895,0.426895,2.3100
1,Balanced Growth,345634.944,0.000000,0.000000,14.5824
2,Protect Whales,10955.880,0.041566,0.041566,0.6300


In [33]:
# 2) Deterministic checks (judgment signals, no LLM)
def euros_per_au(row):
    return float(row['delta_net_eur']) / max(float(row['effort_au']), 1e-6)

def rank_by_eur_per_au(df):
    return sorted([(r['option'], euros_per_au(r)) for _, r in df.iterrows()], key=lambda x: x[1], reverse=True)

def budget_compliant(df, cap):
    return [r['option'] for _, r in df.iterrows() if float(r['effort_au']) <= cap]

def retention_support(df):
    return [r['option'] for _, r in df.iterrows() if float(r['delta_grr_pp']) >= 0 and float(r['delta_nrr_pp']) >= 0]

def churn_shock_risk_flags(df):
    return {r['option']: (float(r['delta_grr_pp']) == 0 and float(r['delta_nrr_pp']) == 0) for _, r in df.iterrows()}

def budget_cut_priority(df, top_k=2):
    eligible = [(r['option'], float(r['effort_au'])) for _, r in df.iterrows()
                if float(r['delta_grr_pp']) >= 0 and float(r['delta_nrr_pp']) >= 0]
    eligible.sort(key=lambda x: x[1])
    return [o for o, _ in eligible[:top_k]]

det_checks = {
    'eur_per_au_rank': rank_by_eur_per_au(scenarios_df),
    'budget_compliant': budget_compliant(scenarios_df, BUDGET_CAP_AU),
    'retention_support': retention_support(scenarios_df),
    'churn_shock_risk_flags': churn_shock_risk_flags(scenarios_df),
    'budget_cut_priority': budget_cut_priority(scenarios_df),
    'greedy_budget_used_au': float(greedy_summary.get('budget_used_au', 0)),
    'greedy_delta_net_eur': float(greedy_summary.get('delta_net_eur', 0)),
    'greedy_delta_grr_pp': float(greedy_summary.get('delta_grr_pp', 0)),
    'constraints_note': f'Engine guardrails enforced upstream (payback <= {MAX_PAYBACK_MONTHS} months).'
}
det_checks

{'eur_per_au_rank': [('Balanced Growth', 23702.198815009866),
  ('Protect Whales', 17390.285714284048),
  ('Retention-first', 9039.974025973961)],
 'budget_compliant': ['Retention-first', 'Balanced Growth', 'Protect Whales'],
 'retention_support': ['Retention-first', 'Balanced Growth', 'Protect Whales'],
 'churn_shock_risk_flags': {'Retention-first': False,
  'Balanced Growth': True,
  'Protect Whales': False},
 'budget_cut_priority': ['Protect Whales', 'Retention-first'],
 'greedy_budget_used_au': 49.99234924999993,
 'greedy_delta_net_eur': 1354803.4239653908,
 'greedy_delta_grr_pp': 0.1483488891239837,
 'constraints_note': 'Engine guardrails enforced upstream (payback <= 18 months).'}

## Design note — why the agent is guided
This agent is constrained by **decision principles**, not decisions.

It mirrors executive reasoning:
- The engine generates options
- The agent applies shared heuristics (risk, budget, robustness)
- The final recommendation remains interpretative, not deterministic


In [34]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


api_key = os.getenv('OPENAI_API_KEY')
llm = ChatOpenAI(api_key=api_key, model=os.getenv('OPENAI_MODEL','gpt-4o-mini'), temperature=0.2)

def tool_get_scenarios(_: str) -> str:
    return scenarios_df.to_csv(index=False)

def tool_get_checks(_: str) -> str:
    import json as _json
    return _json.dumps(det_checks, indent=2)

SYSTEM_PROMPT = '''
You are an Executive Challenger Agent for a B2B SaaS Decision Intelligence project.
You do NOT recompute simulations. You only interpret Decision Engine outputs.

Apply qualitative judgment rules (not hard-coded decisions):
- Zero ΔGRR/ΔNRR implies higher churn exposure
- Under budget cuts, favor low AU + non-degrading retention
- EUR per AU is an efficiency signal, not a forecast

Return a strict, board-ready JSON focused on robustness under constraints.
'''

prompt = (
    # f"{SYSTEM_PROMPT}\n\n"
    # f"Deterministic checks (JSON):\n{tool_get_checks('')}\n\n"
    # f"Scenario table (CSV):\n{tool_get_scenarios('')}\n\n"
    # f"Assume payback gate <= {MAX_PAYBACK_MONTHS} months and budget cap = {BUDGET_CAP_AU} AU."
        'Use get_deterministic_checks first. '
    f'Assume budget cap = {BUDGET_CAP_AU} AU and payback <= {MAX_PAYBACK_MONTHS} months. '
    'Frame the recommendation as most robust under board constraints.'
 )
result = llm.invoke(prompt)
output_text = getattr(result, 'content', str(result))
print(output_text)
result = output_text


To provide a robust recommendation under the specified constraints of a budget cap of 50.0 AU and a payback period of 18 months or less, we will first utilize the `get_deterministic_checks` function to evaluate potential investment opportunities. This function will help us identify projects that meet the financial criteria and align with the board's strategic objectives.

### Step 1: Identify Eligible Projects
Using the `get_deterministic_checks`, we will filter projects based on the following criteria:
- **Budget Cap**: Projects must not exceed a total cost of 50.0 AU.
- **Payback Period**: Projects must have a payback period of 18 months or less.

### Step 2: Analyze Project Viability
Once we have a list of eligible projects, we will analyze their expected returns, risks, and alignment with the company's strategic goals. This analysis will include:
- **Return on Investment (ROI)**: Calculate the expected ROI for each project.
- **Risk Assessment**: Evaluate the risks associated with 

In [35]:
# 4) Save JSON payload for Decision Room / slides
with open('agentic_checks.json','w') as f:
    f.write(result)
print('[OK] Wrote agentic_checks.json')


[OK] Wrote agentic_checks.json
